In [ ]:
import ROOT
import pandas as pd
xs = 4.075 * 1e3 # in fb, see https://twiki.cern.ch/twiki/bin/view/LHCPhysics/LHCHWG136TeVxsec_extrap
# xs = 1.0 # for testing
weights = [
    "hw_nominal",     # nominal MC weight.                  scalar
    "hw_alphaS_up",   # up alpha_s variaton for PHD4LHC;    scalar
    "hw_alphaS_dn",   # down alpha_s variaton for PHD4LHC;  scalar
    "hw_pdf4lhc_unc", # 30 Eigen variation for PHD4LHC      vector
    "hw_qcd",         # muR/muF variation for the given MC; vector
]

In [2]:
dfs = {}
dfs['all'] = ROOT.RDataFrame("mc23e_vbf_hyy", "ntuples/mc23e_vbf_hyy.root")
dfs['300pt'] = dfs['all'].Filter("Higgs_p4.Pt() > 300e3")
dfs['300pt450'] = dfs['all'].Filter("Higgs_p4.Pt() > 300e3 && Higgs_p4.Pt() <= 450e3")
dfs['450pt650'] = dfs['all'].Filter("Higgs_p4.Pt() > 450e3 && Higgs_p4.Pt() <= 650e3")
dfs['650pt'] = dfs['all'].Filter("Higgs_p4.Pt() > 650e3")

len_hw_pdf4lhc_unc = set(dfs['all'].Range(10).Define('len_hw_pdf4lhc_unc', 'hw_pdf4lhc_unc.size()').AsNumpy(['len_hw_pdf4lhc_unc'])['len_hw_pdf4lhc_unc'])
len_hw_qcd         = set(dfs['all'].Range(10).Define('len_hw_qcd', 'hw_qcd.size()').AsNumpy(['len_hw_qcd'])['len_hw_qcd'])
assert (len(len_hw_pdf4lhc_unc) == 1 and len(len_hw_qcd) == 1)
len_hw_pdf4lhc_unc = list(len_hw_pdf4lhc_unc)[0]
len_hw_qcd         = list(len_hw_qcd)[0]

In [3]:
dfs['all'].Filter("hw_alphaS_up == hw_alphaS_up", 'hw_alphaS_up_not_nan').Report().Print()

hw_alphaS_up_not_nan: pass=5430000    all=5430000    -- eff=100.00 % cumulative eff=100.00 %


In [4]:
weight_dict = {}
futures = []
for slice, df in dfs.items():
    weight_dict[slice] = {}
    for weight in weights:
        if weight == "hw_pdf4lhc_unc":
            for i in range(len_hw_pdf4lhc_unc):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        elif weight == "hw_qcd":
            for i in range(len_hw_qcd):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        else:
            weight_dict[slice][weight] = df.Filter(f"{weight} == {weight}").Sum(weight)
            futures.append(weight_dict[slice][weight])
ROOT.RDF.RunGraphs(futures)

1

In [5]:
for slice, weight_sum_dict in weight_dict.items():
    for weight_name, weight_sum in weight_sum_dict.items():
        weight_dict[slice][weight_name] = weight_sum.GetValue()

In [15]:
pdf = pd.DataFrame(weight_dict)
pdf = pdf.apply(lambda row : (row / row['all']), axis=1)
ratio_pdf = pdf.apply(lambda row : row / pdf.iloc[0], axis=1)
pdf.columns = [col + '_acc' for col in pdf.columns]
pdf[[col.split('_')[0] + '_xs' for col in pdf.columns]] = pdf.apply(lambda row : row * xs, axis=1)
pdf = pd.concat([pdf, ratio_pdf.add_suffix('_ratio')], axis=1)
pdf.to_csv("results/mc23e_vbf_hyy.csv", index=True)

In [16]:
pdf

,all_acc,300pt_acc,300pt450_acc,450pt650_acc,650pt_acc,all_xs,300pt_xs,300pt450_xs,450pt650_xs,650pt_xs,all_ratio,300pt_ratio,300pt450_ratio,450pt650_ratio,650pt_ratio
hw_nominal,1.0,0.012896,0.010855,0.001745,0.000296,4075.0,52.551550,44.234208,7.111132,1.206210,1.0,1.000000,1.000000,1.000000,1.000000
hw_alphaS_up,1.0,0.012865,0.010831,0.001739,0.000295,4075.0,52.425995,44.136912,7.086987,1.202096,1.0,0.997611,0.997800,0.996605,0.996589
hw_alphaS_dn,1.0,0.012946,0.010895,0.001754,0.000298,4075.0,52.756709,44.395337,7.147535,1.213837,1.0,1.003904,1.003643,1.005119,1.006323
hw_pdf4lhc_unc_0,1.0,0.012896,0.010855,0.001745,0.000296,4075.0,52.551550,44.234208,7.111132,1.206210,1.0,1.000000,1.000000,1.000000,1.000000
hw_pdf4lhc_unc_1,1.0,0.012891,0.010851,0.001742,0.000298,4075.0,52.532542,44.219583,7.100075,1.212884,1.0,0.999638,0.999669,0.998445,1.005533
hw_pdf4lhc_unc_2,1.0,0.012888,0.010846,0.001745,0.000297,4075.0,52.520376,44.197778,7.111214,1.211384,1.0,0.999407,0.999176,1.000012,1.004289
hw_pdf4lhc_unc_3,1.0,0.012927,0.010873,0.001752,0.000301,4075.0,52.676422,44.307963,7.141052,1.227407,1.0,1.002376,1.001667,1.004208,1.017573
hw_pdf4lhc_unc_4,1.0,0.012889,0.010849,0.001744,0.000295,4075.0,52.521366,44.211432,7.106204,1.203730,1.0,0.999426,0.999485,0.999307,0.997944
hw_pdf4lhc_unc_5,1.0,0.012873,0.010838,0.001740,0.000295,4075.0,52.459208,44.164917,7.090403,1.203889,1.0,0.998243,0.998434,0.997085,0.998075
hw_pdf4lhc_unc_6,1.0,0.012911,0.010865,0.001747,0.000298,4075.0,52.611090,44.274953,7.120059,1.216078,1.0,1.001133,1.000921,1.001255,1.008181
